# Notebook 36 — Input data for reproducing RMM (WH04) and BSISO (Lee et al. 2013)

Downloads **the same data sources the two papers used**, as small OPeNDAP subsets from NOAA PSL, and saves
them to Drive for nb37 (RMM) and nb38 (BSISO). Plan: conversation log, Session 64; decisions: Session 65.

| field | paper | source here | subset saved |
|---|---|---|---|
| OLR | both: NOAA interpolated daily OLR (Liebmann & Smith 1996), 2.5° | PSL `interp_OLR/olr.day.mean.nc` | 15S-15N band (all lons) and 10S-40N, 40-160E |
| u850, u200 | WH04: NCEP-NCAR Reanalysis 1 | PSL `ncep.reanalysis/Dailies/pressure/uwnd.YYYY.nc` | 15S-15N band, 850 + 200 hPa |
| u850 | Lee: NCEP-DOE Reanalysis 2 | PSL `ncep.reanalysis2/Dailies/pressure/uwnd.YYYY.nc` | 10S-40N, 40-160E, 850 hPa |

**Coverage (checked 2026-10-07):**
- The PSL OLR file runs from 1974-06-01 to **2022-12-31**. The days 1978-03-17 to 1978-12-31 are a gap stored as missing values; this matches the 290 missing days in the BoM RMM file.
- The reproduced indices therefore end on 2022-12-31. NCEP R1/R2 go on to 2024, but they are only downloaded up to 2022.
- Downloads cover the whole year, every year: the 120-day running mean and the annual-cycle fit both need all months, including for the BSISO.

Each year is cached on Drive as soon as it is fetched, so a re-run only downloads what is missing.
Runtime: about 15 min on the first run.

## Cell 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, subprocess
import numpy as np
import pandas as pd
try:
    import netCDF4  # noqa: F401  (OPeNDAP access for xarray)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'netCDF4'], check=True)
import xarray as xr

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
IDX_DIR     = f'{PROJECT_DIR}/data/index_repro'
CACHE_DIR   = f'{IDX_DIR}/yearly_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

PSL = 'https://psl.noaa.gov/thredds/dodsC/Datasets'
OLR_URL = f'{PSL}/interp_OLR/olr.day.mean.nc'
R1_URLS = [f'{PSL}/ncep.reanalysis/Dailies/pressure/uwnd.{{y}}.nc',      # current PSL layout
           f'{PSL}/ncep.reanalysis.dailyavgs/pressure/uwnd.{{y}}.nc']   # older layout (fallback)
R2_URLS = [f'{PSL}/ncep.reanalysis2/Dailies/pressure/uwnd.{{y}}.nc',
           f'{PSL}/ncep.reanalysis2.dailyavgs/pressure/uwnd.{{y}}.nc']

OLR_YEARS = range(1978, 2023)   # RMM needs late 1978 for the 120-day window; PSL OLR ends 2022-12-31
R1_YEARS  = range(1978, 2023)
R2_YEARS  = range(1979, 2023)   # BSISO EOF period 1981-2010; earlier years for the 120-day window and margin

BAND = dict(lat=slice(15, -15))                                   # PSL latitudes run 90 -> -90
ASM  = dict(lat=slice(40, -10), lon=slice(40, 160))
print('Output dir:', IDX_DIR)

## Cell 2 — Download helpers (retry + per-year cache)

In [ ]:
def open_dap(urls, retries=4):
    """Open the first URL that works; retry transient THREDDS errors."""
    last = None
    for attempt in range(retries):
        for u in urls:
            try:
                return xr.open_dataset(u)
            except Exception as e:          # noqa: BLE001  (network / server errors)
                last = e
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f'could not open any of {urls}: {last}')

def cached(name, fetch):
    """Load CACHE_DIR/name.npz if present, else call fetch() -> dict of arrays and save it."""
    p = f'{CACHE_DIR}/{name}.npz'
    if os.path.exists(p):
        return dict(np.load(p, allow_pickle=True))
    for attempt in range(4):
        try:
            out = fetch()
            break
        except Exception as e:              # noqa: BLE001
            if attempt == 3:
                raise
            print(f'      retry {name}: {type(e).__name__}')
            time.sleep(10 * (attempt + 1))
    np.savez_compressed(p, **out)
    return out

def as_dict(da):
    return {'values': da.values.astype(np.float32),
            'dates':  pd.DatetimeIndex(da['time'].values).strftime('%Y-%m-%d').values,
            'lat': da['lat'].values.astype(np.float32), 'lon': da['lon'].values.astype(np.float32)}

def stack(parts, key='values'):
    v = np.concatenate([p[key] for p in parts], axis=0)
    d = np.concatenate([p['dates'] for p in parts])
    return v, d
print('Helpers ready.')

## Cell 3 — NOAA interpolated OLR (both indices)

In [ ]:
ds_olr = open_dap([OLR_URL])
band_parts, asm_parts = [], []
t0 = time.time()
for y in OLR_YEARS:
    sel = ds_olr['olr'].sel(time=slice(f'{y}-01-01', f'{y}-12-31'))
    band_parts.append(cached(f'olr_band_{y}', lambda: as_dict(sel.sel(**BAND).load())))
    asm_parts.append(cached(f'olr_asm_{y}',  lambda: as_dict(sel.sel(**ASM).load())))
    print(f'   OLR {y}  ({time.time() - t0:.0f} s)', end='\r')
olr_band, d_olr = stack(band_parts)
olr_asm, _      = stack(asm_parts)
np.savez_compressed(f'{IDX_DIR}/noaa_olr_15S15N.npz', olr=olr_band, dates=d_olr,
                    lat=band_parts[0]['lat'], lon=band_parts[0]['lon'])
np.savez_compressed(f'{IDX_DIR}/noaa_olr_asm.npz', olr=olr_asm, dates=d_olr,
                    lat=asm_parts[0]['lat'], lon=asm_parts[0]['lon'])
gap = np.isnan(olr_band).all(axis=(1, 2))
print(f'\nOLR band {olr_band.shape}, ASM {olr_asm.shape}, {d_olr[0]} .. {d_olr[-1]}')
print(f'   all-missing days: {int(gap.sum())} ({d_olr[gap][0] if gap.any() else "-"} .. {d_olr[gap][-1] if gap.any() else "-"})')
print(f'   value range: {np.nanmin(olr_band):.0f} .. {np.nanmax(olr_band):.0f} W m-2')

## Cell 4 — NCEP-NCAR Reanalysis 1: u850 and u200, 15S-15N (RMM)

In [ ]:
r1_parts = []
t0 = time.time()
for y in R1_YEARS:
    def fetch(y=y):
        da = open_dap([u.format(y=y) for u in R1_URLS])['uwnd'].sel(level=[850, 200], **BAND).load()
        return as_dict(da)
    r1_parts.append(cached(f'r1_uwnd_band_{y}', fetch))
    print(f'   R1 {y}  ({time.time() - t0:.0f} s)', end='\r')
u_r1, d_r1 = stack(r1_parts)
np.savez_compressed(f'{IDX_DIR}/ncep_r1_uwnd_15S15N.npz', u850=u_r1[:, 0], u200=u_r1[:, 1], dates=d_r1,
                    lat=r1_parts[0]['lat'], lon=r1_parts[0]['lon'])
print(f'\nR1 u {u_r1.shape} (time, level[850,200], lat, lon), {d_r1[0]} .. {d_r1[-1]}; '
      f'NaN {int(np.isnan(u_r1).sum())}; u850 {np.nanmin(u_r1[:, 0]):.0f}..{np.nanmax(u_r1[:, 0]):.0f} m/s')

## Cell 5 — NCEP-DOE Reanalysis 2: u850, 10S-40N, 40-160E (BSISO)

In [ ]:
r2_parts = []
t0 = time.time()
for y in R2_YEARS:
    def fetch(y=y):
        return as_dict(open_dap([u.format(y=y) for u in R2_URLS])['uwnd'].sel(level=850, **ASM).load())
    r2_parts.append(cached(f'r2_u850_asm_{y}', fetch))
    print(f'   R2 {y}  ({time.time() - t0:.0f} s)', end='\r')
u_r2, d_r2 = stack(r2_parts)
np.savez_compressed(f'{IDX_DIR}/ncep_r2_u850_asm.npz', u850=u_r2, dates=d_r2,
                    lat=r2_parts[0]['lat'], lon=r2_parts[0]['lon'])
print(f'\nR2 u850 {u_r2.shape}, {d_r2[0]} .. {d_r2[-1]}; NaN {int(np.isnan(u_r2).sum())}; '
      f'grid lat {r2_parts[0]["lat"][0]:.1f}..{r2_parts[0]["lat"][-1]:.1f}, lon {r2_parts[0]["lon"][0]:.1f}..{r2_parts[0]["lon"][-1]:.1f}')

---
## Done

Saved to `data/index_repro/`:
- `noaa_olr_15S15N.npz` and `noaa_olr_asm.npz`
- `ncep_r1_uwnd_15S15N.npz`
- `ncep_r2_u850_asm.npz`

The per-year pieces stay in `yearly_cache/`. Next: **nb37** (RMM) and **nb38** (BSISO).

**Checks:**
- **Cell 3:** about 290 all-missing OLR days, 1978-03-17 to 1978-12-31.
- **Cell 5:** the grid should read 40..-10 latitude by 40..160 longitude (21 × 49 points).